<a href="https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
--
I use a Decision Tree because this lane predicts whether a page's impressions decline from March to April.

A Decision Tree is appropriate because it can capture simple non-linear relationships between March search signals and the decline outcome, while remaining interpretable.

The model uses only March features: impressions, clicks, and average position. No April outcome information is used as a feature.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score

features = [
    "march_impressions",
    "march_clicks",
    "march_position"
]

target = "target_decline"

print("Features:", features)
print("Target:", target)


Features: ['march_impressions', 'march_clicks', 'march_position']
Target: target_decline


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
--
I use a grouped client-level split so that pages from the same client do not appear in both training and test data.

This is more honest than a random row split because pages belonging to the same client can share similar behavior.

The test set contains clients that were not present in the training set.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

X = model_df[features].copy()
y = model_df[target].copy()
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.22,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(model_df.iloc[train_idx]["client_hash_id"])
test_clients = set(model_df.iloc[test_idx]["client_hash_id"])

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Shared clients:", len(train_clients & test_clients))


Train rows: 133490
Test rows: 43247
Train clients: 36
Test clients: 11
Shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
--
I train the Decision Tree on the training clients and evaluate it on the held-out clients.

The comparison uses Average Precision, matching the Week-4 baseline metric on the same test data.

The baseline is the transparent Week-4 rule, while the Decision Tree is the learned model.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score
import pandas as pd

model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

model_ap = average_precision_score(
    y_test,
    model_scores
)

baseline_ap = 0.448416

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Decision Tree"
    ],
    "Average Precision": [
        baseline_ap,
        model_ap
    ]
})

print(comparison)

            Method  Average Precision
0  Week-4 baseline           0.448416
1    Decision Tree           0.478791


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
--
The model makes both false-positive and false-negative errors, so the predictions should be treated as decision-support rather than certainty.

The observed feature importance shows that March clicks contribute the most to the tree, followed by March position and March impressions.

The model is therefore relying mainly on existing search engagement and visibility signals.

A recommendation can still be wrong because a page may decline for reasons that are not represented by these three March signals.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix

predicted_labels = (model_scores >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    predicted_labels
).ravel()

print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(feature_importance)


True negatives: 12820
False positives: 10774
False negatives: 10091
True positives: 9562
             feature  importance
1       march_clicks    0.520634
2     march_position    0.322411
0  march_impressions    0.156954


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.